# SynBioCrow 2.4.18 — Internal-Anchor Structural Near-Miss Audit
Upload only the 2.4.18 SOURCE ZIP. The final target is excluded from scoring. This diagnoses structural proximity to literature-supported internal pathway anchors on development records only.


In [ ]:
from google.colab import drive,files
from pathlib import Path
import hashlib,json,os,shutil,subprocess,sys,time,zipfile
drive.mount("/content/drive",force_remount=False);T0=time.time()
BASE=Path("/content/drive/MyDrive/SynBioCrow/2.4");REPO=Path("/content/SynBioCrow_2_4_18");SRC=Path("/content/src2418");RUN=Path("/content/synbiocrow_2_4_18");OUT=BASE/"internal_anchor_audit_2_4_18";OUT.mkdir(parents=True,exist_ok=True)
def st(p,m):print(f"[2.4.18] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {m}",flush=True)
def run(cmd,cwd=None,timeout=1800):
 e=os.environ.copy()
 if cwd:e["PYTHONPATH"]=str(cwd)+os.pathsep+e.get("PYTHONPATH","")
 print("$"," ".join(map(str,cmd)),flush=True);q=subprocess.run(list(map(str,cmd)),cwd=cwd,env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout);print(q.stdout,flush=True)
 if q.returncode:raise subprocess.CalledProcessError(q.returncode,cmd,output=q.stdout)
st(3,"upload 2.4.18 SOURCE ZIP only");u=files.upload()
if len(u)!=1:raise RuntimeError("Upload exactly one SOURCE ZIP")
name=next(iter(u));z=Path("/content")/name;z.write_bytes(u[name]);print("SOURCE SHA256",hashlib.sha256(z.read_bytes()).hexdigest())
st(8,"clone develop/2.4");shutil.rmtree(REPO,ignore_errors=True);run(["git","clone","--depth","1","--branch","develop/2.4","https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
st(14,"overlay source");shutil.rmtree(SRC,ignore_errors=True);SRC.mkdir(parents=True)
with zipfile.ZipFile(z) as q:q.extractall(SRC)
for p in SRC.rglob("*"):
 if p.is_file() and p.name not in {"README_SOURCE.txt","SOURCE_MANIFEST.json"}:
  d=REPO/p.relative_to(SRC);d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,d)
st(20,"install core");run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=900)
man=json.loads((REPO/"benchmarks/2.4/split_manifest_2_4_11.json").read_text());dev=[r["record_id"] for r in man["records"] if r["split"]=="development"];sealed=[r["record_id"] for r in man["records"] if r["split"]!="development"]
st(28,"discover development artifacts + 2.4.16 ranking + 2.4.17 PubChem cache")
dirs=sorted([p for p in BASE.glob("development_routes_*") if p.is_dir()],key=lambda p:p.stat().st_mtime,reverse=True);inp=next((d for d in dirs[:20] if all((d/f"{r}.json").is_file() for r in dev)),None)
if inp is None:raise FileNotFoundError("complete development artifacts not found")
ranked=BASE/"native_discrimination_2_4_16"/"native_enriched_ranked_routes.json";cache=BASE/"development_truth_2_4_17"/"pubchem_anchor_cache.json"
if not ranked.is_file():raise FileNotFoundError(ranked)
st(40,"run target-excluded internal-anchor audit");shutil.rmtree(RUN,ignore_errors=True);RUN.mkdir(parents=True)
cmd=[sys.executable,str(REPO/"scripts/v24_audit_internal_anchors_2_4_18.py"),"--input-dir",str(inp),"--ranked-json",str(ranked),"--truth",str(REPO/"benchmarks/2.4/development_truth_anchors_2_4_17.json"),"--split-manifest",str(REPO/"benchmarks/2.4/split_manifest_2_4_11.json"),"--out-dir",str(RUN)]
if cache.is_file():cmd+=["--cache",str(cache)]
run(cmd,cwd=REPO)
st(72,"verify firewall");s=json.loads((RUN/"internal_anchor_near_miss_summary.json").read_text());assert s["target_excluded_from_scoring"] and not s["ranking_tuned"] and not s["generation_invoked"] and not s["validation_truth_accessed"] and not s["evaluation_truth_accessed"];print(json.dumps(s,indent=2))
st(86,"persist + bundle")
for p in RUN.iterdir():
 if p.is_file():shutil.copy2(p,OUT/p.name)
bundle=OUT/"SynBioCrow_2_4_18_INTERNAL_ANCHOR_NEAR_MISS_AUDIT_ALL_OUTPUTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as q:
 for p in sorted(OUT.iterdir()):
  if p.is_file() and p!=bundle:q.write(p,p.name)
st(100,"PASS");print("BUNDLE",bundle);print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest());files.download(str(bundle))


## Rescue download


In [ ]:
from google.colab import files
from pathlib import Path
p=Path("/content/drive/MyDrive/SynBioCrow/2.4/internal_anchor_audit_2_4_18/SynBioCrow_2_4_18_INTERNAL_ANCHOR_NEAR_MISS_AUDIT_ALL_OUTPUTS.zip")
assert p.is_file(),p
files.download(str(p))
